## PART 1: Local LLM Initialization & Baseline Test

In [1]:
import requests
import json

# Config
OLLAMA_URL = "http://localhost:11434/api/generate"
GEMMA_MODEL = "llama3.2:1b"

def ask_local_llm(prompt: str, model: str = GEMMA_MODEL) -> str:
    """
    Sends a prompt to the local Ollama model via its REST API.
    Returns the generated response as a single string.
    """
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False  # Disable streaming to receive the full response at once
    }
    
    try:
        response = requests.post(OLLAMA_URL, json=payload)
        
        # Check for HTTP errors
        if response.status_code != 200:
            raise RuntimeError(f"Ollama API error {response.status_code}: {response.text}")
            
        data = response.json()
        return data.get("response", "").strip()
        
    except requests.exceptions.ConnectionError:
        return "[ERROR] Could not connect to Ollama. Is the service running on localhost:11434?"

# Baseline Test
print(f"Connecting to {GEMMA_MODEL} via {OLLAMA_URL}...")

# We ask a simple question that the LLM should answer natively
test_question = "Who are you? Answer in one short sentence."
print(f"\n[Question] {test_question}")

answer = ask_local_llm(test_question)
print(f"\n[LLM Answer] {answer}")

Connecting to llama3.2:1b via http://localhost:11434/api/generate...

[Question] Who are you? Answer in one short sentence.

[LLM Answer] I am an artificial intelligence designed to assist and communicate with users.

You can think of me as a computer program that learns and improves over time, allowing me to provide information, answer questions, and have conversations with people like you.


## PART 2: Loading Knowledge Graph & Building Schema Summary

In [2]:
import rdflib
from rdflib import Graph

# File path to the knowledge graph
GRAPH_FILE = "expanded.nt"

# Constraints to avoid overloading the LLM's prompt context limit
MAX_PREDICATES = 20
MAX_CLASSES = 40
SAMPLE_TRIPLES = 5

def load_graph(file_path: str) -> Graph:
    """Loads the RDF graph from the specified N-Triples file."""
    print(f"Loading graph from {file_path}... (This might take a few seconds)")
    g = Graph()
    g.parse(file_path, format="nt") 
    print(f"[OK] Loaded {len(g)} triples.")
    return g

def get_prefix_block() -> str:
    """Provides standard prefixes to help the LLM generate valid SPARQL."""
    defaults = {
        "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
        "xsd": "http://www.w3.org/2001/XMLSchema#",
        "owl": "http://www.w3.org/2002/07/owl#",
        "wd": "http://www.wikidata.org/entity/",
        "wdt": "http://www.wikidata.org/prop/direct/"
    }
    lines = [f"PREFIX {p}: <{ns}>" for p, ns in defaults.items()]
    return "\n".join(sorted(lines))

def list_distinct_predicates(g: Graph, limit=MAX_PREDICATES) -> list:
    """Extracts unique predicates used in the graph."""
    q = f"SELECT DISTINCT ?p WHERE {{ ?s ?p ?o . }} LIMIT {limit}"
    return [str(row.p) for row in g.query(q)]

def list_distinct_classes(g: Graph, limit=MAX_CLASSES) -> list:
    """Extracts unique classes (rdf:type) used in the graph."""
    q = f"SELECT DISTINCT ?cls WHERE {{ ?s a ?cls . }} LIMIT {limit}"
    return [str(row.cls) for row in g.query(q)]

def sample_triples(g: Graph, limit=SAMPLE_TRIPLES) -> list:
    """Extracts a few sample triples to show the LLM the exact data structure."""
    q = f"SELECT ?s ?p ?o WHERE {{ ?s ?p ?o . }} LIMIT {limit}"
    return [(str(r.s), str(r.p), str(r.o)) for r in g.query(q)]

def build_schema_summary(g: Graph) -> str:
    """Constructs the prompt summary that will be injected into the LLM context."""
    prefixes = get_prefix_block()
    preds = list_distinct_predicates(g)
    clss = list_distinct_classes(g)
    samples = sample_triples(g)
    
    pred_lines = "\n".join(f"- {p}" for p in preds)
    cls_lines = "\n".join(f"- {c}" for c in clss)
    sample_lines = "\n".join(f"- {s} {p} {o}" for s, p, o in samples)
    
    summary = f"""
{prefixes}

# Predicates (sampled, unique up to {MAX_PREDICATES})
{pred_lines}

# Classes / rdf:type (sampled, unique up to {MAX_CLASSES})
{cls_lines}

# Sample triples (up to {SAMPLE_TRIPLES})
{sample_lines}
"""
    return summary.strip()

# --- Execution ---
kg_graph = load_graph(GRAPH_FILE)
schema_summary = build_schema_summary(kg_graph)

print("\nSchema Summary Generated:")
print("Here is a preview of the context the LLM will use:")
print("-" * 50)
print(schema_summary[:500] + "\n\n... [TRUNCATED] ...")
print("-" * 50)

Loading graph from expanded.nt... (This might take a few seconds)
[OK] Loaded 53568 triples.

Schema Summary Generated:
Here is a preview of the context the LLM will use:
--------------------------------------------------
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

# Predicates (sampled, unique up to 20)
- http://www.wikidata.org/prop/P12695
- http://www.wikidata.org/prop/direct/P355
- http://www.wikidata.org/prop/statement/P19
- http://www.wikidata.org/prop/di

... [TRUNCATED] ...
--------------------------------------------------


## PART 3: NL to SPARQL Generation & Execution

In [4]:
import re

# Instructions for the LLM
SPARQL_INSTRUCTIONS = """
You are an expert SPARQL generator. Convert the user QUESTION into a valid SPARQL 1.1 SELECT query for the given RDF graph schema.
Follow strictly:
- Use ONLY the IRIs/prefixes visible in the SCHEMA SUMMARY.
- Prefer readable SELECT projections with variable names.
- Do NOT invent new predicates/classes.
- Return ONLY the SPARQL query in a single fenced code block.
- No explanations or extra text outside the code block.
"""

def make_sparql_prompt(schema_summary: str, question: str) -> str:
    """Builds the prompt combining instructions, schema, and the user's question [cite: 186-191]."""
    return f"""{SPARQL_INSTRUCTIONS}

SCHEMA SUMMARY:
{schema_summary}

QUESTION:
{question}

Return only the SPARQL query in a code block.
"""

# Parsing the LLM Output
CODE_BLOCK_RE = re.compile(r"`{3}(?:sparql)?\s*(.*?)`{3}", re.IGNORECASE | re.DOTALL)

def extract_sparql_from_text(text: str) -> str:
    """Extracts the first code block content; fallback to whole text if no block is found."""
    m = CODE_BLOCK_RE.search(text)
    if m:
        return m.group(1).strip()
    return text.strip()

def generate_sparql(question: str, schema_summary: str) -> str:
    """Calls the local LLM to generate the SPARQL query."""
    prompt = make_sparql_prompt(schema_summary, question)
    raw_response = ask_local_llm(prompt) 
    return extract_sparql_from_text(raw_response)


'''
def generate_sparql(question: str, schema_summary: str) -> str:
    q_lower = question.lower()
    
    if "microsoft" in q_lower:
        return """
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT ?p ?o WHERE { 
            ?s rdfs:label ?l . 
            FILTER(CONTAINS(LCASE(?l), "microsoft")) 
            ?s ?p ?o . 
        } LIMIT 5
        """
    
    prompt = make_sparql_prompt(schema_summary, question)
    return extract_sparql_from_text(ask_local_llm(prompt))
    '''
# --- 3. Executing SPARQL on your Graph ---
def run_sparql(g, query: str):
    """Executes the SPARQL query on the RDFLib graph and returns variables and rows."""
    res = g.query(query)
    vars_ = [str(v) for v in res.vars]
    rows = [tuple(str(cell) for cell in r) for r in res]
    return vars_, rows

# --- 4. Self-Repair Mechanism ---
REPAIR_INSTRUCTIONS = """
The previous SPARQL query failed to execute. Using the SCHEMA SUMMARY and the ERROR MESSAGE, return a corrected SPARQL 1.1 SELECT query.
Follow strictly:
- Use only known prefixes/IRIs.
- Keep it as simple and robust as possible.
- Return ONLY a single code block with the corrected SPARQL.
"""

def repair_sparql(schema_summary: str, question: str, bad_query: str, error_msg: str) -> str:
    """Asks the LLM to fix its own broken SPARQL query based on the error output."""
    prompt = f"""{REPAIR_INSTRUCTIONS}

SCHEMA SUMMARY:
{schema_summary}

ORIGINAL QUESTION:
{question}

BAD SPARQL:
{bad_query}

ERROR MESSAGE:
{error_msg}

Return only the corrected SPARQL in a code block.
"""
    raw_response = ask_local_llm(prompt)
    return extract_sparql_from_text(raw_response)

# --- 5. Main Orchestration Function ---
def answer_with_sparql_generation(g, schema_summary: str, question: str, try_repair: bool = True) -> dict:
    """Generates, executes, and potentially repairs a SPARQL query to answer a natural language question."""
    sparql = generate_sparql(question, schema_summary)
    
    try:
        vars_, rows = run_sparql(g, sparql)
        return {"query": sparql, "vars": vars_, "rows": rows, "repaired": False, "error": None}
    
    except Exception as e:
        err = str(e)
        if try_repair:
            print(f"    [!] SPARQL Syntax Error detected. Triggering LLM self-repair...")
            repaired_query = repair_sparql(schema_summary, question, sparql, err)
            try:
                vars_, rows = run_sparql(g, repaired_query)
                return {"query": repaired_query, "vars": vars_, "rows": rows, "repaired": True, "error": None}
            except Exception as e2:
                return {"query": repaired_query, "vars": [], "rows": [], "repaired": True, "error": str(e2)}
        else:
            return {"query": sparql, "vars": [], "rows": [], "repaired": False, "error": err}

print("[OK] All RAG logic functions are defined and ready.")

[OK] All RAG logic functions are defined and ready.


## Part 4

In [5]:
# Baseline function
def answer_no_rag(question: str) -> str:
    """Baseline answer without using the Knowledge Graph ."""
    prompt = f"Answer the following question as best as you can:\n\n{question}"
    return ask_local_llm(prompt)

# Display function
def pretty_print_result(result: dict):
    """Formats the output of the SPARQL execution for readability."""
    if result.get("error"):
        print(f"\n[Execution Error] {result['error']}")
        
    print("\n[SPARQL Query Used]")
    print("-" * 50)
    print(result["query"])
    print("-" * 50)
    
    print(f"\n[Repaired?] {result['repaired']}")
    
    vars_ = result.get("vars", [])
    rows = result.get("rows", [])
    
    if not rows:
        print("\n[Results] No rows returned by the query.")
        return
        
    print("\n[Results]")
    print(" | ".join(vars_))
    print("-" * 50)
    for r in rows[:20]:
        print(" | ".join(r))
        
    if len(rows) > 20:
        print(f"... (showing 20 of {len(rows)} rows) [cite: 284]")

# --- 3. Interactive CLI Loop ---
print("\n" + "="*60)
print("🤖 KNOWLEDGE GRAPH RAG CHATBOT READY")
print("Type 'quit' or 'exit' to stop the chatbot.")
print("="*60)

# Main interaction loop [cite: 288-297]
while True:
    user_q = input("\nQuestion: ").strip()
    
    if user_q.lower() in ["quit", "exit"]:
        print("Exiting chatbot. See you!")
        break
        
    if not user_q:
        continue
        
    print("\n--- 1. Baseline Answer (No RAG, raw LLM knowledge) ---")
    print(answer_no_rag(user_q))
    
    print("\n--- 2. SPARQL-generation RAG (Llama 3.2 + RDFLib) ---")
    # kg_graph and schema_summary are variables generated in Cell 2
    rag_result = answer_with_sparql_generation(kg_graph, schema_summary, user_q, try_repair=True)
    pretty_print_result(rag_result)


🤖 KNOWLEDGE GRAPH RAG CHATBOT READY
Type 'quit' or 'exit' to stop the chatbot.



Question:  What are the labels related to microsoft?



--- 1. Baseline Answer (No RAG, raw LLM knowledge) ---
Microsoft has several well-known product labels across its various divisions. Here are some of them:

1. **Windows**: The Windows label is used for the operating system, including versions like Windows 10 and Windows 11.
2. **Office**: Microsoft Office is a suite of productivity software that includes Word, Excel, PowerPoint, and Outlook. The Office label covers these applications.
3. **Surface**: The Surface label refers to Microsoft's line of tablets, laptops, and other products designed for computing on the go.
4. **Xbox**: The Xbox label is used for Microsoft's gaming console family, including the original Xbox, Xbox 360, Xbox One, and Xbox Series X/S consoles.
5. **Azure**: Azure is a cloud computing platform offered by Microsoft, which provides services like storage, computing power, and data analytics.
6. **Edge**: Edge is a web browser developed by Microsoft, known for its fast loading speeds and integration with the Windo


Question:  exit


Exiting chatbot. See you!
